# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and process the [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema at: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata
print("Dataset Name:", dataset.metadata.name)
print("Description:\n", dataset.metadata.description)


## 2. Data Overview
Explore the available record sets, fields, and their `@id`s.

Let's programmatically discover and print the available record sets in this Croissant package, along with their field `@id`s, using the `@id` property for referencing.

In [ ]:
# List all record sets and their fields by @id
record_sets_metadata = dataset.metadata.record_sets
if not record_sets_metadata:
    print("No record sets are listed in the metadata. Trying to enumerate using dataset API...")
    # Try to discover record sets via implementation
    record_sets = dataset.record_sets()
    print("Discovered record sets (by @id):")
    for rs in record_sets:
        print("- @id:", rs['@id'], "name:", rs.get('name', ''))
        if 'fields' in rs:
            print("  fields @id list:", [f['@id'] for f in rs['fields']])
else:
    for rs in record_sets_metadata:
        print("Record set @id:", rs.id)
        print("   Name:", getattr(rs, 'name', ''))
        print("   Field @ids:", [f.id for f in rs.fields])
        print()

### Sample Record Inspection
Let's print a sample record from each record set.

**Note**: Replace `<record_set_id>` with the chosen `@id` for the mass spectrometry main table.

In [ ]:
# Try to list records from the main record set
from pprint import pprint
all_record_sets = list(dataset.record_sets())
if not all_record_sets:
    raise RuntimeError("No record sets found in this Croissant dataset.")

# For demonstration, pick the first record set
main_record_set = all_record_sets[0]['@id']
print(f"Using record set: {main_record_set}\nSample record:")
sample_recs_iter = dataset.records(record_set=main_record_set)
for idx, rec in enumerate(sample_recs_iter):
    pprint(rec)
    if idx >= 2:
        break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis, using the discovered record set and field `@id`s.


In [ ]:
# Collect all record sets @id
record_set_ids = [rs['@id'] for rs in all_record_sets]

# Read all tables into DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}:", df.columns.tolist())
        print(df.head(2))
    else:
        print(f"No records found for {record_set_id}.")


## 4. Exploratory Data Analysis (EDA)
Now let's conduct basic EDA on one record set. 

We will select a numeric field, filter for records above a threshold, normalize the field, and optionally group by a categorical field.

Choose your target numeric field (by `@id`) and a group-by field as detected above.

In [ ]:
# Select main record set and column for numeric EDA
main_df = dataframes[main_record_set]
print("--- Main record set columns ---")
print(main_df.columns.tolist())

# Let's auto-detect a numeric field (float/int columns)
numeric_candidates = main_df.select_dtypes(include=['number']).columns.tolist()
if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Detected numeric field: {numeric_field}")
else:
    numeric_field = main_df.columns[0]
    print(f"No numeric fields auto-detected, using first column: {numeric_field}")

# Set a filtering threshold
threshold = main_df[numeric_field].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field]) else 0
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (mean): {len(filtered_df)} records")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Optionally group by a categorical field
group_candidates = main_df.select_dtypes(include=['object', 'category']).columns.tolist()
group_field = group_candidates[0] if group_candidates else None
if group_field:
    print(f"Grouping by {group_field} and averaging numeric fields...")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(grouped_df.head())
else:
    print("No categorical group field detected.")

## 5. Visualization
Let's visualize the distribution of the numeric field and the grouped means (if available).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution histogram
plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field], kde=True, bins=30)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If group_field exists, plot group means
if group_field is not None:
    group_means = main_df.groupby(group_field)[numeric_field].mean()
    group_means.plot(kind='bar', figsize=(9,4), title=f'{numeric_field} mean by {group_field}')
    plt.ylabel(f'Mean {numeric_field}')
    plt.show()

## 6. Conclusion
We demonstrated the usage of the `mlcroissant` library to programmatically discover, extract, and explore the FAIR² Croissant dataset on mass spectrometry protein abundance. Using only the Croissant schema, we:
- Inspected its data structure (`@id` driven reference to record sets and fields),
- Loaded tabular data into DataFrames,
- Performed basic filtering, normalization, and groupings by column,
- Visualized numeric value distributions.

You can now apply further analyses or machine learning using the loaded DataFrames.
